In [42]:
#import libraries
#tkinter-python built in GUI library
#PIL (pillow) used fro image processing
#Numpy is used to work with images as numerical arrays
#opencv for image processing
#os proviing functions for working with files and folders
#joblib used to load previously trained Ml model (KNN model .sav file)
import tkinter as tk
from PIL import ImageTk,Image,ImageDraw
import numpy as np
import cv2
import os
import joblib

#loads KNN model
model=joblib.load('sinhala-character-knn.sav')

#define canvas sizes
width=500
height=500 

#creatre label dictionary (KNN model doesnt directly return sinhala characters)
label_dict={0:'අ',1:'එ',2:'ඉ',3:'උ'}

#font button + prediction label style
font_btn='Helvetica 20 bold'
font_label='Helverica 22 bold'

#create the main tkinter window
win=tk.Tk()

#functions
#function called evrytime when user moves mouse while holding left mouse button
def event_function(event):

    #x and y cordinates
    x=event.x
    y=event.y
    
    x1=x-30
    x2=x+30

    y1=y-30
    y2=y+30

    # We create a 60 × 60 area around the mouse position.
    
    # For example, if:

    # x = 250
    # y = 300
    
    # then:

    # x1 = 220
    # x2 = 280
    # y1 = 270
    # y2 = 330
    
    # This gives a circular brush approximately 60 pixels in diameter

    canvas.create_oval((x1,y1,x2,y2),fill='black')
    # draws a black circle on the Tkinter canvas.
    img_draw.ellipse((x1,y1,x2,y2),fill='black')
    # draw the same black circle onto our PIL image.
    #store 2 versions of drawing

    # 1. canvas
    #    → visual drawing shown to the user
    
    # 2. img
    #    → actual image stored in memory


#save function
def save():
    global count

    #convert the PIL image into a NumPy array (opencv works with numpy arrays)
    img_array=np.array(img) 
    #create path for image save
    path=os.path.join('data',str(count)+'.jpg')
    #path=data/0.jpg

    # saves the NumPy image array as a JPG file.
    cv2.imwrite(path,img_array)
    #prevent the next saved image form replacing the previous one
    count=count+1

def clear():
    global img,img_draw

    #remove everything currently drawn on the Tkinter canvas
    canvas.delete('all')
    #create a new blank PIL image
    img=Image.new('RGB', (width,height),(255,255,255))
    #create a new image object
    img_draw=ImageDraw.Draw(img)
    #reset prediction label, after clicking clear-GUI shows none
    label_status.config(text='PREDICTED CHARACTER:  NONE')

#predict function
def predict():

    # The overall process is:

    # Drawing
    #    ↓
    # PIL Image
    #    ↓
    # NumPy Array
    #    ↓
    # Grayscale
    #    ↓
    # Resize to 8 × 8
    #    ↓
    # Flatten to 1 × 64
    #    ↓
    # KNN model
    #    ↓
    # Numerical class
    #    ↓
    # Sinhala character


    #same like model training process (grayscale+resize+flattern)
    img_array=np.array(img)
    img_array=cv2.cvtColor(img_array,cv2.COLOR_BGR2GRAY) #converting in to gray image
    img_array=cv2.resize(img_array,(8,8)) #resize into 8*8
    img_array=np.reshape(img_array,(1,64)) #reshape into 1*64 - 1D

    result=model.predict(img_array)[0]
    #print(label_dict[result])
    label=label_dict[result]
    label_status.config(text='PREDICTED CHARACTER:'+label)

#create drawing canvas
canvas=tk.Canvas(win,width=width,height=height,bg='white')
canvas.grid(row=0,column=0,columnspan=4)
# Place the canvas using Tkinter's grid layout.

#row=0:
#  First row.

#column=0:
#  Start from first column.

#columnspan=4:
#  Canvas occupies all 4 columns.

#save button
button_save=tk.Button(win,text='SAVE',bg='green',fg='white',font=font_btn,command=save)
button_save.grid(row=1,column=0)

#predict button
button_predict=tk.Button(win,text='PREDICT',bg='blue',fg='white',font=font_btn,command=predict)
button_predict.grid(row=1,column=1)

#clear button
button_clear=tk.Button(win,text='CLEAR',bg='yellow',fg='white',font=font_btn, command=clear)
button_clear.grid(row=1,column=2)

#exit button
button_exit=tk.Button(win,text='EXIT',bg='red',fg='white',font=font_btn,command=win.destroy)
button_exit.grid(row=1,column=3)

#prediction label status
label_status=tk.Label(win,text='PREDICTED CHARACTER: NONE', bg='white', font=font_label)
# columnspan=4 it covers all 4 columns
label_status.grid(row=2,column=0,columnspan=4)

#connect mouse movement to drawing function
canvas.bind('<B1-Motion>', event_function)

#create the actual PIL image
img=Image.new('RGB',(width,height),(255,255,255))
#Create an ImageDraw object connected to "img".
img_draw=ImageDraw.Draw(img)


win.mainloop()